# MiDaS CNN Pipeline
**Ivan Martín Campoy | UAB, 2025/26**

This notebook runs the MiDaS CNN-based depth estimation model across all experimental
datasets, serving as a cross-architecture comparison against the ViT-based Depth Anything
models in `tfg_pipeline.ipynb`.

MiDaS (`Intel/dpt-hybrid-midas`) uses a DPT (Dense Prediction Transformer) head on top
of a hybrid CNN-ViT backbone trained on multiple datasets simultaneously. Unlike the
Depth Anything family, it does not use semi-supervised training on unlabelled data.

All paths are relative to the `notebooks/` folder.
Results are written to `../results/depth_map_comparisons/midas/`,
`../results/roi_analysis/midas/`, and `../results/psychophysics/midas/`.

Sections:
1. Imports and Environment Setup
2. Model Loading
3. Inference and Depth Map Generation
4. ROI Analysis (Depth Gap Measurement)
5. PSE and JND Analysis
6. Video Export

## 1. Imports and Environment Setup

In [1]:
#Libraries

from PIL import Image 
import matplotlib.pyplot as plt 
import cv2
import os 
import re #regular expressions
import torch
import numpy as np
import sys
import math
from PIL import ImageFilter
from transformers import pipeline
import pandas as pd

## 2. Model Loading

MiDaS is loaded through the HuggingFace `transformers` pipeline using the
`depth-estimation` task. The `dpt-hybrid-midas` checkpoint uses a ViT-Hybrid backbone
(ResNet-50 + ViT) with a DPT head, trained on a mix of ReDWeb, DIML, MegaDepth,
WSVD, TartanAir and other datasets.

The `size` parameter is left at the default 384px to match the resolution used
in the Depth Anything runs and keep comparisons consistent.

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu" #set device as GPU if available
print(f"Using Device: {device}")

Using Device: cuda


In [3]:
#loading model/s
models = {
    "MiDaS": "Intel/dpt-hybrid-midas" #load midas cnn baseline from hugging face hub
}

pipelines = {}
print("\nLoading CNN models")
for version, repo in models.items():
    try:
        #initialize the depth estimation pipeline for the cnn architecture
        pipelines[version] = pipeline(task="depth-estimation", model=repo, device=device, torch_dtype=torch.float32)
        print(f" {version} loaded successfully.")
    except Exception as e:
        print(f" Couldn't Load {version}. Error: {e}")


Loading CNN models


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/414 [00:00<?, ?it/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


 MiDaS loaded successfully.


## 3. Inference and Depth Map Generation

### 3.1 Core functions: `compare_single_image` and `process_dataset`

These functions mirror the design of the main pipeline. `compare_single_image` produces
a two-panel output (original + MiDaS depth map) instead of four panels, since only one
model is evaluated here.

One difference from the Depth Anything pipeline: MiDaS consistently outputs higher
values for closer objects (same convention as V1 and V2), so no depth map inversion
is needed.

In [10]:
def compare_single_image(image_path, save_path=None):
    """
    runs the available cnn model (midas) and plots it alongside the original image
    """
    try:
        available_models = sorted(pipelines.keys())
        if not available_models: return

        #setup naming
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        clean_name = re.sub(r'\d+', '', base_name).replace('_', ' ').strip().title()

        img_pil = Image.open(image_path).convert("RGB")
        
        results = {}
        for version in available_models:
            model = pipelines[version]
            #midas pipeline returns the standard dictionary structure like depth anything v1/v2
            res = model(img_pil, timeout=None)
            results[version] = res["predicted_depth"].squeeze().cpu().numpy()

        num_plots = len(results) + 1
        fig, axes = plt.subplots(1, num_plots, figsize=(5 * num_plots, 5))
        if num_plots == 1: axes = [axes]
        fig.suptitle(clean_name, fontsize=14, fontweight='bold', y=0.98)

        #0. original image
        axes[0].imshow(img_pil)
        axes[0].set_title("Original Image", fontsize=10, fontweight='bold')
        axes[0].axis('off')

        #1-n. depth maps
        for i, version in enumerate(available_models):
            depth_map_ptr = np.array(results[version])
            
            #basic normalization: 0 to 1
            d_min, d_max = depth_map_ptr.min(), depth_map_ptr.max()
            depth_norm = (depth_map_ptr - d_min) / (d_max - d_min + 1e-8)

            #midas output usually represents closer objects with higher values (matches v1/v2)
            axes[i+1].imshow(depth_norm, cmap='inferno', interpolation="bilinear")
            axes[i+1].set_title(f"{version} CNN", fontsize=10, fontweight='bold')
            axes[i+1].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.92])

        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()

    except Exception as e:
        print(f" error in compare_single_image: {e}")

def process_dataset(input_folder="3D_illusions", output_folder="results_cnn"):
    """
    automates processing for the entire directory structure targeting the cnn pipeline
    """
    #final check before starting
    print(f"models currently loaded: {list(pipelines.keys())}")
    
    total = 0
    if not os.path.exists(input_folder):
        print(f" error: input folder '{input_folder}' not found.")
        return

    for root, dirs, files in os.walk(input_folder):
        for filename in files:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, filename)
                
                #maintain subdirectory structure
                rel_path = os.path.relpath(root, input_folder)
                target_dir = os.path.join(output_folder, rel_path)
                os.makedirs(target_dir, exist_ok=True)
                
                save_path = os.path.join(target_dir, f"cmp_{filename}")
                
                print(f" processing -> {filename}")
                compare_single_image(img_path, save_path=save_path)
                total += 1

    print(f"dataset processing complete. total images: {total}")

### 3.2 Batch processing over all datasets

Each `process_dataset` call walks the input folder recursively and saves a `cmp_*.png`
comparison for every image found, preserving the subfolder structure in the output.

The four datasets covered are:
- **Baseline dataset** — the curated 3D illusion collection
- **Synthetic** — the fifteen Corridor Illusion ablation variants
- **Minecraft** — the voxel laboratory experiments
- **Psychophysics** — the 30-frame stimulus sequences (all 15 experiments)

In [ ]:
#Experiments:

process_dataset(input_folder="../data/dataset", output_folder="../results/depth_map_comparisons/midas/baseline_dataset") #for the original 3D illusions dataset
process_dataset(input_folder="../data/synthetic/corridor_illusion", output_folder="../results/depth_map_comparisons/midas/synthetic") #for the original 3D illusions dataset
process_dataset(input_folder="../data/minecraft", output_folder="../results/depth_map_comparisons/midas/minecraft") #Minecraft Experiments
process_dataset(input_folder="../data/psychophysics", output_folder="../results/depth_map_comparisons/midas/psychophysics") #Psychophysics Experiments


models currently loaded: ['MiDaS']
 processing -> contrast_000.png
 processing -> contrast_008.png
 processing -> contrast_016.png
 processing -> contrast_024.png
 processing -> contrast_033.png
 processing -> contrast_041.png
 processing -> contrast_049.png
 processing -> contrast_057.png
 processing -> contrast_066.png
 processing -> contrast_074.png
 processing -> contrast_082.png
 processing -> contrast_091.png
 processing -> contrast_099.png
 processing -> contrast_107.png
 processing -> contrast_115.png
 processing -> contrast_124.png
 processing -> contrast_132.png
 processing -> contrast_140.png
 processing -> contrast_148.png
 processing -> contrast_157.png
 processing -> contrast_165.png
 processing -> contrast_173.png
 processing -> contrast_182.png
 processing -> contrast_190.png
 processing -> contrast_198.png
 processing -> contrast_206.png
 processing -> contrast_215.png
 processing -> contrast_223.png
 processing -> contrast_231.png
 processing -> contrast_240.png
 proc

## 4. ROI Analysis: Depth Gap Measurement

### 4.1 Bounding-box tool adapted for a single model

This is the same interactive ROI tool used in the main pipeline, adapted for a
two-panel comparison (original + MiDaS) instead of four panels (original + V1 + V2 + V3).

The tool opens each `cmp_*.png` comparison in a resizable OpenCV window. The user
draws a bounding box on the first panel (the original image); the same box is
mirrored automatically onto the MiDaS panel at the correct pixel offset.

For each ROI, the tool records `mean_intensity` (a proxy for predicted depth) and,
for every second ROI in a pair, `gap_with_prev_roi` (the depth gap delta_I between
the two ROIs). Results are appended to `tfg_roi_metrics.csv`, using the same column
format as the main pipeline so both CSVs can be compared directly.

**Controls:** drag to draw a box, `s` to save and move to the next image,
`r` to reset boxes on the current image, `n` to skip, `Esc` to exit.

Note: this cell requires an active display and cannot run headlessly.

In [ ]:
import cv2
import numpy as np
import os
import csv

rectangles = []
drawing = False
ix, iy = -1, -1
img_with_commands = None
raw_img = None
scale_ratio = 1.0
display_offset = 0
orig_offset = 0
banner_h = 40
num_panels = 2  # original + MiDaS only
last_category = ""

def draw_rect(event, x, y, flags, param):
    """handles click-and-drag to draw bounding boxes."""
    global ix, iy, drawing, img_with_commands, rectangles, display_offset, banner_h, num_panels

    if event == cv2.EVENT_LBUTTONDOWN:
        if y < banner_h: return  # ignore clicks on banner
        if x > display_offset: return  # force drawing on panel 1 only
        drawing = True
        ix, iy = x, y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            temp_img = img_with_commands.copy()
            for i in range(num_panels):
                nx1, nx2 = ix + i * display_offset, x + i * display_offset
                cv2.rectangle(temp_img, (nx1, iy), (nx2, y), (0, 255, 255), 1)
            cv2.imshow("tfg_lab_midas", temp_img)

    elif event == cv2.EVENT_LBUTTONUP:
        if drawing:
            drawing = False
            x1, x2 = min(ix, x), max(ix, x)
            y1, y2 = min(iy, y), max(iy, y)

            if x2 - x1 > 2 and y2 - y1 > 2:
                rectangles.append((x1, y1, x2, y2))
                for i in range(num_panels):
                    nx1, nx2 = x1 + i * display_offset, x2 + i * display_offset
                    cv2.rectangle(img_with_commands, (nx1, y1), (nx2, y2), (0, 255, 0), 2)
                    cv2.putText(img_with_commands, f"roi{len(rectangles)}", (nx1, y1-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                cv2.imshow("tfg_lab_midas", img_with_commands)

def run_midas_roi_experiment(results_path="../results/depth_map_comparisons/midas/baseline_dataset",
                              output_path="../results/roi_analysis/midas/baseline_dataset"):
    """
    Same ROI extraction logic as the main pipeline's run_tfg_experiment, but adapted
    for the two-panel MiDaS comparisons (original + MiDaS) instead of four panels.
    """
    global rectangles, img_with_commands, raw_img, scale_ratio, display_offset, orig_offset, banner_h, num_panels, last_category

    os.makedirs(output_path, exist_ok=True)
    csv_file = os.path.join(output_path, "tfg_roi_metrics.csv")

    if not os.path.exists(csv_file):
        with open(csv_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["category", "image", "model", "roi_id", "area_px", "mean_intensity", "gap_with_prev_roi"])

    for category in sorted(os.listdir(results_path)):
        cat_path = os.path.join(results_path, category)
        if not os.path.isdir(cat_path): continue

        if category != last_category:
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["---", f"CATEGORY: {category.upper()}", "---", "---", "---", "---", "---"])
            last_category = category

        valid_ext = ('.png', '.jpg', '.jpeg', '.bmp')
        image_files = [f for f in os.listdir(cat_path) if f.lower().endswith(valid_ext)]

        for filename in image_files:
            print(f"\n--- analyzing: {filename} ---")

            target_path = os.path.join(cat_path, filename)
            raw_img = cv2.imread(target_path, cv2.IMREAD_GRAYSCALE)
            if raw_img is None: continue

            h, w_total = raw_img.shape

            # auto-detect offset logic (same as main pipeline, generalised to num_panels)
            analysis_region = raw_img[int(h*0.15):int(h*0.95), :]
            _, thresh = cv2.threshold(analysis_region, 240, 255, cv2.THRESH_BINARY_INV)
            col_sums = np.sum(thresh, axis=0)
            content_cols = np.where(col_sums > 20)[0]

            if len(content_cols) > 0:
                jumps = np.where(np.diff(content_cols) > 15)[0]
                clusters = []
                start_idx = 0
                for j in jumps:
                    clusters.append((content_cols[start_idx], content_cols[j]))
                    start_idx = j + 1
                clusters.append((content_cols[start_idx], content_cols[-1]))

                if len(clusters) >= num_panels:
                    centers = [(start + end) / 2.0 for start, end in clusters]
                    orig_offset = int(np.mean(np.diff(centers[:num_panels])))
                else:
                    orig_offset = w_total // num_panels
            else:
                orig_offset = w_total // num_panels

            max_w, max_h = 1800, 800
            scale_ratio = min(max_w/w_total, max_h/h, 1.0)
            new_w, new_h = int(w_total * scale_ratio), int(h * scale_ratio)
            display_offset = int(orig_offset * scale_ratio)

            base_display = cv2.cvtColor(cv2.resize(raw_img, (new_w, new_h)), cv2.COLOR_GRAY2BGR)

            img_with_commands = np.zeros((new_h + banner_h, new_w, 3), dtype=np.uint8)
            img_with_commands[banner_h:, :] = base_display

            cv2.putText(img_with_commands, "keys: [s] save | [r] reset | [n] next  --  DRAG to draw rectangles on panel 1",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            rectangles = []
            cv2.namedWindow("tfg_lab_midas", cv2.WINDOW_AUTOSIZE)
            cv2.setMouseCallback("tfg_lab_midas", draw_rect)

            break_all = False
            while True:
                cv2.imshow("tfg_lab_midas", img_with_commands)
                key = cv2.waitKey(1) & 0xFF

                if key == ord('s'):
                    out_img = cv2.cvtColor(raw_img, cv2.COLOR_GRAY2BGR)

                    with open(csv_file, mode='a', newline='') as f:
                        writer = csv.writer(f)
                        means = {"orig": [], "midas": []}

                        for i, (px1, py1, px2, py2) in enumerate(rectangles):
                            true_y1, true_y2 = py1 - banner_h, py2 - banner_h
                            ox1, oy1 = int(px1 / scale_ratio), int(true_y1 / scale_ratio)
                            ox2, oy2 = int(px2 / scale_ratio), int(true_y2 / scale_ratio)

                            area = (ox2 - ox1) * (oy2 - oy1)

                            roi_orig = raw_img[oy1:oy2, ox1:ox2]
                            roi_midas = raw_img[oy1:oy2, ox1 + orig_offset : ox2 + orig_offset]

                            mean_orig = round(np.mean(roi_orig), 2)
                            mean_midas = round(np.mean(roi_midas), 2)

                            means["orig"].append(mean_orig)
                            means["midas"].append(mean_midas)

                            gap_orig = round(abs(mean_orig - means["orig"][i-1]), 2) if i % 2 == 1 else 0.0
                            gap_midas = round(abs(mean_midas - means["midas"][i-1]), 2) if i % 2 == 1 else 0.0

                            writer.writerow([category, filename, "orig", i+1, area, mean_orig, gap_orig])
                            writer.writerow([category, filename, "midas", i+1, area, mean_midas, gap_midas])

                            for j, m_val in enumerate([mean_orig, mean_midas]):
                                curr_x1 = ox1 + j * orig_offset
                                curr_x2 = ox2 + j * orig_offset
                                cv2.rectangle(out_img, (curr_x1, oy1), (curr_x2, oy2), (0, 255, 0), 2)
                                cv2.putText(out_img, f"r{i+1}: {m_val}", (curr_x1, oy1-10),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    out_dir = os.path.join(output_path, category)
                    os.makedirs(out_dir, exist_ok=True)
                    cv2.imwrite(os.path.join(out_dir, f"analisis_{filename}"), out_img)
                    print("  -> saved successfully.")
                    break

                elif key == ord('r'):
                    img_with_commands[banner_h:, :] = base_display
                    img_with_commands[:banner_h, :] = 0
                    cv2.putText(img_with_commands, "keys: [s] save | [r] reset | [n] next  --  DRAG to draw rectangles on panel 1",
                                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                    rectangles = []

                elif key == ord('n'):
                    break

                elif key == 27:
                    break_all = True
                    break

            if break_all: break
        if break_all: break

    cv2.destroyAllWindows()

# run_midas_roi_experiment(results_path="../results/depth_map_comparisons/midas/baseline_dataset", output_path="../results/roi_analysis/midas/baseline_dataset")
# run_midas_roi_experiment(results_path="../results/depth_map_comparisons/midas/synthetic/", output_path="../results/roi_analysis/midas/synthetic/")
# run_midas_roi_experiment(results_path="../results/depth_map_comparisons/midas/minecraft", output_path="../results/roi_analysis/midas/minecraft")


--- analyzing: cmp_base_image.png ---
  -> saved successfully.

--- analyzing: cmp_cube_walls.jpg ---
  -> saved successfully.

--- analyzing: cmp_cylinder_pattern.png ---
  -> saved successfully.

--- analyzing: cmp_cylinder_pattern_bordered.png ---
  -> saved successfully.

--- analyzing: cmp_high-low_filter.png ---
  -> saved successfully.

--- analyzing: cmp_high_pass.png ---
  -> saved successfully.

--- analyzing: cmp_low_pass.png ---
  -> saved successfully.

--- analyzing: cmp_minecraft_version.png ---
  -> saved successfully.

--- analyzing: cmp_no_cyllinders.png ---
  -> saved successfully.

--- analyzing: cmp_no_walls.png ---
  -> saved successfully.

--- analyzing: cmp_only_cylinders.png ---
  -> saved successfully.

--- analyzing: cmp_shepard_illusion.jpg ---
  -> saved successfully.

--- analyzing: cmp_size_prior.jpg ---
  -> saved successfully.

--- analyzing: cmp_solid.png ---
  -> saved successfully.

--- analyzing: cmp_solid_bordered.png ---
  -> saved successfully.


### 4.2 ROI analysis on psychophysics sequences

Run the same tool on the MiDaS psychophysics comparison panels, producing the
data needed for the PSE/JND analysis in Section 5.

In [ ]:
run_midas_roi_experiment(results_path="../results/depth_map_comparisons/midas/psychophysics", output_path="../results/roi_analysis/midas/psychophysics")

## 5. PSE and JND Analysis

### 5.1 Psychometric curve fitting for MiDaS

This mirrors `analyze_all_models_comparison` from the main pipeline, but plots a
single model (MiDaS) instead of V1/V2/V3, using the CSV produced in Section 4.2.

The same thresholds are extracted via linear interpolation:

- **PSE:** the stimulus step where delta_I crosses zero
- **JND:** the distance from the PSE to the point where |delta_I| reaches 84%
  of its maximum value (equivalent to one standard deviation of a cumulative
  Gaussian)

`run_full_midas_analysis` iterates over all experiment categories found in the
CSV and calls `analyze_midas_comparison` for each one.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def analyze_midas_comparison(csv_path, category_name, threshold_pct=0.84):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"error: file {csv_path} not found")
        return

    df = df[df['category'] != '---']
    df['mean_intensity'] = pd.to_numeric(df['mean_intensity'], errors='coerce')
    df['roi_id'] = pd.to_numeric(df['roi_id'], errors='coerce')

    df['step'] = df['image'].str.extract(r'_(\d+)\.')[0].astype(float)
    df = df.dropna(subset=['step', 'mean_intensity'])
    df['step'] = df['step'].astype(int)

    cat_df = df[df['category'].str.strip() == category_name].copy()
    cat_df = cat_df.drop_duplicates(subset=['model', 'step', 'roi_id'], keep='last')

    if cat_df.empty: return

    color = '#d62728'  # distinct colour from V1/V2/V3 palette

    plt.figure(figsize=(12, 8))

    model_df = cat_df[cat_df['model'] == 'midas']
    if model_df.empty:
        print(f"no midas data found for category {category_name}")
        return

    r1 = model_df[model_df['roi_id'] == 1].sort_values('step')
    r2 = model_df[model_df['roi_id'] == 2].sort_values('step')

    if not r1.empty and not r2.empty:
        steps = r1['step'].values
        y_diff = r1.set_index('step')['mean_intensity'] - r2.set_index('step')['mean_intensity']
        y_values = y_diff.values

        line, = plt.plot(steps, y_values, color=color, linewidth=2, marker='o', alpha=0.8)

        # pse (crossover at zero)
        pse_x = None
        for i in range(len(steps) - 1):
            if y_values[i] * y_values[i+1] <= 0:
                weight = abs(y_values[i]) / (max(abs(y_values[i]) + abs(y_values[i+1]), 1e-6))
                pse_x = steps[i] + weight * (steps[i+1] - steps[i])
                plt.scatter([pse_x], [0], color=color, s=80, edgecolors='white', zorder=5)
                break

        # jnd (distance from pse to threshold)
        jnd_val = None
        max_delta = np.max(np.abs(y_values))
        threshold_target = max_delta * threshold_pct

        for i in range(len(steps) - 1):
            if pse_x is not None and steps[i] >= (pse_x - 1):
                if abs(y_values[i]) <= threshold_target <= abs(y_values[i+1]):
                    weight = (threshold_target - abs(y_values[i])) / (max(abs(y_values[i+1]) - abs(y_values[i]), 1e-6))
                    threshold_step = steps[i] + weight * (steps[i+1] - steps[i])
                    jnd_val = threshold_step - pse_x

                    plt.hlines(y=y_values[i+1] if y_values[i+1] > 0 else -threshold_target,
                               xmin=pse_x, xmax=threshold_step, color=color, linestyle=':', alpha=0.5)
                    plt.annotate('', xy=(threshold_step, 0), xytext=(pse_x, 0),
                                 arrowprops=dict(arrowstyle='<->', color=color, lw=1.5))
                    break

        pse_str = f"{pse_x:.2f}" if pse_x is not None else "n/a"
        jnd_str = f"{jnd_val:.2f}" if jnd_val is not None else "n/a"
        line.set_label(f'MiDaS | PSE: {pse_str} | \u2194 JND: {jnd_str}')

    plt.axhline(y=0, color='black', linewidth=1.2, alpha=0.5)
    plt.title(f'PSYCHOPHYSICAL ANALYSIS (MiDaS): {category_name.upper()}\nJND Threshold: {threshold_pct*100:.0f}%', fontsize=14)
    plt.xlabel('Stimulus Evolution (Step)')
    plt.ylabel('Depth Delta (Target - Context)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    plt.grid(True, linestyle=':', alpha=0.3)

    base_dir = "../results/psychophysics/midas"
    exp_dir = os.path.join(base_dir, category_name)
    if not os.path.exists(exp_dir): os.makedirs(exp_dir)

    save_path = os.path.join(exp_dir, f"psychophysics_midas_{category_name}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"saved successfully in: {save_path}")

def run_full_midas_analysis(csv_path):
    try:
        df = pd.read_csv(csv_path)
        categories = df[df['category'] != '---']['category'].unique()
        for cat in categories:
            analyze_midas_comparison(csv_path, cat)
    except Exception as e:
        print(f"error during analysis: {e}")

#to run the script:
run_full_midas_analysis("../results/roi_analysis/midas/psychophysics/tfg_roi_metrics.csv")

error during analysis: [Errno 2] No such file or directory: '../results/roi_analysis/midas/psychophysics/tfg_roi_metrics.csv'


## 6. Video Export

Psychophysics sequences are exported as MP4 videos at 15 fps for use in the
project defence presentation. This uses the same `create_experiment_videos`
function as the main pipeline, reading from `../data/psychophysics/` where
the original stimulus frames are stored.

In [17]:
def create_experiment_videos(target_directory="../data/psychophysics", fps=15):
    """
    #generates lightweight mp4 videos for each experiment folder.
    #optimized for presentations with small file sizes and high compatibility.
    """
    #check if the provided target directory exists on the system
    if not os.path.exists(target_directory):
        #display error message if the path is invalid
        print(f"error: target directory {target_directory} not found")
        return

    #iterate through all subdirectories in the target path, sorted alphabetically
    for folder_name in sorted(os.listdir(target_directory)):
        #construct the full path to the current experiment folder
        folder_path = os.path.join(target_directory, folder_name)
        
        #verify that the current item is a directory and not a file
        if os.path.isdir(folder_path):
            #define a tuple of supported image file extensions
            valid_ext = ('.png', '.jpg', '.jpeg')
            #collect and sort all image files to ensure correct frame sequence
            image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(valid_ext)])
            
            #skip the current folder if no valid images are found inside
            if not image_files:
                #move to the next iteration in the loop
                continue
            
            #determine video dimensions using the first image in the list
            first_frame_path = os.path.join(folder_path, image_files[0])
            #load the image into memory using opencv
            first_frame = cv2.imread(first_frame_path)
            #handle cases where the image file might be corrupted or unreadable
            if first_frame is None: continue
            
            #extract the height, width, and channels from the image shape
            h, w, _ = first_frame.shape
            
            #set the output filename using the folder name as a base
            video_name = f"{folder_name}_summary.mp4"
            #join the filename with the folder path for local storage
            video_path = os.path.join(folder_path, video_name)
            
            #define the fourcc codec as mp4v for high compression and compatibility
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            #initialize the videowriter object with the specified parameters
            video_writer = cv2.VideoWriter(video_path, fourcc, fps, (w, h))
            
            #inform the user about the current video being generated
            print(f"compiling {video_name}...")
            
            #loop through the sorted list of image files to build the video
            for img_name in image_files:
                #load the individual frame file
                frame = cv2.imread(os.path.join(folder_path, img_name))
                #ensure the frame was loaded correctly before writing
                if frame is not None:
                    #append the current frame to the video file
                    video_writer.write(frame)
            
            #close the videowriter and finalize the mp4 file properly
            video_writer.release()
            #confirm successful saving of the video file
            print(f"-> saved: {video_path}")

#standard boilerplate for direct script execution
if __name__ == "__main__":
    #execute the video generation function with default settings
    #ensure your psychophysics_experiments folder is in the same directory
    create_experiment_videos()

compiling aerial_perspective_summary.mp4...
-> saved: Psychophysics_results\aerial_perspective\aerial_perspective_summary.mp4
compiling blur_bias_summary.mp4...
-> saved: Psychophysics_results\blur_bias\blur_bias_summary.mp4
compiling ebbinghaus_bias_summary.mp4...
-> saved: Psychophysics_results\ebbinghaus_bias\ebbinghaus_bias_summary.mp4
compiling figure_ground_bias_summary.mp4...
-> saved: Psychophysics_results\figure_ground_bias\figure_ground_bias_summary.mp4
compiling height_bias_summary.mp4...
-> saved: Psychophysics_results\height_bias\height_bias_summary.mp4
compiling hollow_face_bias_summary.mp4...
-> saved: Psychophysics_results\hollow_face_bias\hollow_face_bias_summary.mp4
compiling horizon_bias_summary.mp4...
-> saved: Psychophysics_results\horizon_bias\horizon_bias_summary.mp4
compiling kanizsa_bias_summary.mp4...
-> saved: Psychophysics_results\kanizsa_bias\kanizsa_bias_summary.mp4
compiling occlusion_bias_summary.mp4...
-> saved: Psychophysics_results\occlusion_bias\occl